In [ ]:
# Cell 0
import os
from pathlib import Path

current_dir = Path.cwd()
data_dir = None

check_dir = current_dir
while check_dir != check_dir.parent:
    if "satria-data-bdc" in check_dir.name.lower():
        data_dir = check_dir
        break
    check_dir = check_dir.parent

if data_dir is None:
    data_dir = current_dir

# Derive subdirs
train_dir = data_dir / "train"
test_dir  = data_dir / "test"

submission_path = data_dir / "submission.csv"

# Sanity checks
assert train_dir.exists(), f"train_dir not found: {train_dir}"
assert test_dir.exists(),  f"test_dir not found:  {test_dir}"
assert submission_path.exists(), f"submission.csv not found: {submission_path}"

print(f"data_dir  : {data_dir}")
print(f"train_dir : {train_dir}  ({len(list(train_dir.glob('*/*')))} files)")
print(f"test_dir  : {test_dir}  ({len(list(test_dir.glob('*')))} files)")
print(f"submission: {submission_path}  ({submission_path.stat().st_size:,} bytes)")

In [ ]:
# Cell 1 (Directory structure utilities and execution)

def print_tree(root_dir, max_files_per_folder=3, prefix=""):
    root_dir = Path(root_dir)
    entries = sorted(root_dir.iterdir(), key=lambda x: (x.is_file(), x.name))
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        if entry.is_dir():
            n_files = len(list(entry.glob("*")))
            print(f"{prefix}{connector}{entry.name}/  ({n_files} items)")
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, max_files_per_folder, prefix + extension)
        else:
            print(f"{prefix}{connector}{entry.name}")

def summarize_structure(data_dir):
    data_dir = Path(data_dir)
    print(f"Folder structure: {data_dir}\n")
    print(data_dir.name + "/")
    for split_dir in sorted(data_dir.iterdir()):
        if not split_dir.is_dir():
            continue
        print(f"├── {split_dir.name}/")
        subdirs = [d for d in split_dir.iterdir() if d.is_dir()]

        # Show per-class file counts if class subfolders exist, otherwise list flat files
        if subdirs:
            for cls_dir in sorted(subdirs):
                n_files = len(list(cls_dir.glob("*.*")))
                print(f"│   ├── {cls_dir.name}/  -> {n_files} files")
        else:
            files = list(split_dir.glob("*.*"))
            print(f"│   -> {len(files)} files (no subfolders/labels)")
            if files:
                print(f"│   -> example filenames: {[f.name for f in files[:5]]}")

summarize_structure(data_dir)
print_tree(data_dir)

In [ ]:
# Cell 2 (Corrupt image detection for train and test sets)

from PIL import Image

# Scan all images recursively and return corrupt files with their error messages
def find_corrupt_images(folder_path):
    corrupt_files = {}
    all_files = list(Path(folder_path).rglob("*.*"))

    for file_path in all_files:
        try:
            with Image.open(file_path) as img:
                img.verify()
        except Exception as e:
            corrupt_files[str(file_path)] = str(e)

    return corrupt_files, len(all_files)

# Check train split for corrupt images
train_corrupt, train_total = find_corrupt_images(data_dir / "train")
print(f"TRAIN total files scanned: {train_total}")
print(f"TRAIN corrupt files found: {len(train_corrupt)}")
if train_corrupt:
    print("TRAIN corrupt file list:")
    for path, err in train_corrupt.items():
        print(f"  - {path} -> {err}")

# Check test split for corrupt images
test_corrupt, test_total = find_corrupt_images(data_dir / "test")
print(f"\nTEST total files scanned: {test_total}")
print(f"TEST corrupt files found: {len(test_corrupt)}")
if test_corrupt:
    print("TEST corrupt file list:")
    for path, err in test_corrupt.items():
        print(f"  - {path} -> {err}")

In [ ]:
# Cell 3 (Detect duplicate files between train and test splits using MD5 hashing)

import hashlib

# Compute MD5 hash in chunks to handle large files efficiently
def compute_file_hash(file_path, chunk_size=8192):
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()

# Build hash lookup table for all test files
test_hashes = {}  # {hash: file_path}
test_files = list((data_dir / "test").rglob("*.*"))

for file_path in test_files:
    file_hash = compute_file_hash(file_path)
    test_hashes[file_hash] = str(file_path)

print(f"Total unique hashes in test: {len(test_hashes)} (from {len(test_files)} files)")

# Find train files whose hash matches any test file
train_test_duplicates = {}  # {train_file_path: matching_test_file_path}
train_files = list((data_dir / "train").rglob("*.*"))

for file_path in train_files:
    file_hash = compute_file_hash(file_path)
    if file_hash in test_hashes:
        train_test_duplicates[str(file_path)] = test_hashes[file_hash]

print(f"\nTrain files identical (exact match) to test files: {len(train_test_duplicates)}")
if train_test_duplicates:
    print("Duplicate pairs (train -> test):")
    for train_path, test_path in train_test_duplicates.items():
        print(f"  - {train_path}  <->  {test_path}")

In [ ]:
# Cell 4 (Define class folders and sample files per class)

import random
from PIL import Image

random.seed(42)

# Sample up to n_samples files per class from the train directory
def sample_files_per_class(data_dir, class_folders, n_samples=500):
    samples = {}
    for cls in class_folders:
        files = list((data_dir / "train" / cls).rglob("*.*"))
        samples[cls] = random.sample(files, min(n_samples, len(files)))
    return samples

class_folders = ["0_Recyclable", "1_Electronic", "2_Organic"]
sampled_files = sample_files_per_class(data_dir, class_folders, n_samples=500)